# Burrows's Delta

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* Z-scored MFW plus a distance measure; the standard attribution method

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09b_burrows_delta.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Burrows's Delta (2002) is the most-used method in computational authorship
attribution, and it is simple enough to write from scratch in ten lines — which is
what you will do here, rather than calling a library.

Three steps:

1. Take the *n* most frequent words
2. **Z-score each word's rate across the corpus** — so a word varying by 0.1
   counts as much as one varying by 5
3. Measure the distance between documents in that z-scored space

Step 2 is the insight. Without it *the* would drown out every other feature.

## The tooling

**numpy** for the z-scored feature matrices and **scipy** for clustering.
Stylometry's field standard is **stylo** (R), which implements every Delta
variant plus bootstrap consensus trees; **faststylometry** is the Python
equivalent. We write Delta out because it is ten lines.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["n_words"] = works["text"].str.split().str.len()
works = works[works["n_words"] >= 5000].reset_index(drop=True)

MFW = 150
corpus_counts = Counter()
tokenised = []
for t in works["text"]:
    toks = tokenise(t)
    tokenised.append(toks)
    corpus_counts.update(toks)
features = [w for w, _ in corpus_counts.most_common(MFW)]
rates = np.array([[toks.count(w) / len(toks) * 100 for w in features]
                  for toks in tokenised])
print(f"{rates.shape[0]} works × {rates.shape[1]} most-frequent words")

In [ ]:
# Burrows's Delta, in full. This is the entire method.
def delta_matrix(rates):
    z = (rates - rates.mean(axis=0)) / (rates.std(axis=0) + 1e-12)
    n = z.shape[0]
    d = np.zeros((n, n))
    for i in range(n):
        d[i] = np.abs(z - z[i]).mean(axis=1)     # mean absolute z-difference
    return d, z

D, Z = delta_matrix(rates)
print(f"delta matrix {D.shape}, values from {D[D > 0].min():.3f} to {D.max():.3f}")
print()
print("Delta is a DISTANCE: smaller means more similar. It has no absolute")
print("scale — 0.4 is only meaningful relative to the other distances here.")

In [ ]:
QUERY = 0                   # ← change this: index of the work to attribute

order = np.argsort(D[QUERY])
q = works.iloc[QUERY]
print(f"query: {q['title'][:60]}")
print(f"       by {q['author'][:50]}")
print()
print(f"{'delta':>7}  {'author':<30}{'title'}")
print("-" * 88)
for i in order[1:11]:
    r = works.iloc[i]
    mark = " ←" if r["author"] == q["author"] else ""
    print(f"{D[QUERY, i]:>7.3f}  {str(r['author'])[:30]:<30}{str(r['title'])[:40]}{mark}")

## The real test: leave-one-out

Attributing a work whose author you know proves nothing on its own. The proper
evaluation hides the answer: for every work, find its nearest neighbour among the
*others*, and check whether that neighbour is by the same author.

In [ ]:
counts = works["author"].value_counts()
multi = counts[counts >= 2].index
testable = works.index[works["author"].isin(multi)].tolist()

correct = 0
for i in testable:
    order = np.argsort(D[i])
    nearest = next(j for j in order if j != i)
    correct += works.iloc[nearest]["author"] == works.iloc[i]["author"]

print(f"{len(testable)} works have another work by the same author in the corpus")
print(f"nearest neighbour is the same author: {correct}/{len(testable)} "
      f"= {correct / len(testable):.1%}")
print()
n_auth = len(multi)
print(f"chance level is roughly 1/{len(works) - 1:.0f} per guess — so this is")
print("far above chance, which is the claim Delta actually supports.")

In [ ]:
# The standard stylometry figure: a Delta dendrogram.
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

sel = [i for i in works.index if works.iloc[i]["author"] in multi][:26]
sub_D = D[np.ix_(sel, sel)]
np.fill_diagonal(sub_D, 0)
Zl = linkage(squareform(sub_D, checks=False), method="average")

labels = [f"{str(works.iloc[i]['author'])[:20]} — {str(works.iloc[i]['title'])[:26]}"
          for i in sel]
fig, ax = plt.subplots(figsize=(9, 9))
dendrogram(Zl, labels=labels, orientation="left", leaf_font_size=8, ax=ax)
ax.set_xlabel("Burrows's Delta")
ax.set_title(f"Delta dendrogram, {MFW} MFW")
plt.tight_layout(); plt.show()
print("Works by the same author should sit on the same branch. Where they do")
print("not, look at genre and at transcription source before concluding anything.")

### 🔧 Your turn 1

Change `MFW` to 50, 100, 300, 500 and re-run the leave-one-out score.

Plot the accuracy against MFW if you like. The curve is usually flat across a
wide middle range — which is reassuring, and means a result that only appears at
one specific value should be distrusted.

### 🔧 Your turn 2

Replace the distance in `delta_matrix` with cosine distance on the same z-scores
(cosine Delta, Smith & Aldridge 2011).

Does leave-one-out accuracy improve? It usually does, particularly with more
features — which is why cosine Delta has become the common default.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Accuracy typically rises to around 100–150 MFW and then plateaus.
Very low counts lack signal; very high counts admit content words that group texts
by subject instead of author. Report the value and, ideally, show the curve — a
reviewer who sees a flat plateau is much easier to convince than one shown a
single number.

**Your turn 2.** Cosine Delta usually outperforms the original mean-absolute
version, especially above 500 features, because it is insensitive to overall
magnitude and responds to the *profile* of deviations. Implementing it is a
four-line change and gives you a genuine methods comparison to report.

**What Delta cannot do.** It ranks candidates; it does not decide. There is no
threshold below which a match is "proven", and a work whose true author is absent
from your candidate set will still produce a confident-looking nearest neighbour.
Attribution claims need external evidence as well — which is where the historian
in you takes over from the code.

</details>

## Where this stops

You can run and evaluate the standard attribution method. Next: **09c**, applying
it to a question that is not about authorship at all.